# MAMRL: GPU training and restart in Colab
Select **Runtime → Change runtime type → GPU**. This notebook runs the repository implementation.
Upload a source ZIP for uncommitted work, or select a GitHub revision containing GPU support.
The default job is a small connectivity check. Increase the settings only after its tests pass.
No GPU performance result is implied by the presence of this notebook.

In [ ]:
from pathlib import Path
import os, subprocess, sys, zipfile, shutil
SOURCE_MODE = "upload"  # "upload" or "github"
REPOSITORY = "https://github.com/Kim-YoungIn/MultiAgent_MetaRL_Project.git"
REVISION = ""  # explicit revision containing GPU support; required for github mode
ROOT = Path("/content/mamrl_source")
if not ROOT.exists():
    if SOURCE_MODE == "upload":
        from google.colab import files
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one source ZIP built as described in README")
        ROOT.mkdir()
        with zipfile.ZipFile(archives[0]) as z:
            for item in z.infolist():
                if not (ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()):
                    raise ValueError("Invalid archive path")
            z.extractall(ROOT)
    elif SOURCE_MODE == "github":
        if not REVISION.strip():
            raise ValueError("Set REVISION explicitly before cloning")
        subprocess.run(["git", "clone", REPOSITORY, str(ROOT)], check=True)
        subprocess.run(["git", "checkout", REVISION], cwd=ROOT, check=True)
    else:
        raise ValueError("Unknown SOURCE_MODE")
if not (ROOT / "05_toy_models/mamrl/devices.py").exists():
    raise RuntimeError("Source does not contain the GPU implementation")
os.chdir(ROOT)
os.environ["PYTHONPATH"] = str(ROOT / "05_toy_models")
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)

In [ ]:
import torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA build:", torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable: select a GPU runtime or explicitly choose CPU elsewhere")
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run([sys.executable, "-B", "-m", "mamrl.diagnose_device", "--smoke", "--require-cuda"], check=True)
for pattern in ("test_device_models.py", "test_meta_acceleration.py"):
    subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], check=True)

## Persistent backup
Working files live on the VM. Checkpoints and completed artifacts are copied to a separate Drive
run folder at checkpoint boundaries. Restore the whole folder, including `sources`, `protocol.json`
and `manifest.json`. Resume requires the same source/settings/PyTorch/device type.
A runtime image change can prevent exact resume; retain the environment details and portable weights.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
RUN_NAME = "mamrl_gpu_smoke_001"  # unique name for a new experiment
LOCAL_RUN = Path("/content/mamrl_runs") / RUN_NAME
DRIVE_RUN = Path("/content/drive/MyDrive/MAMRL") / RUN_NAME
RESUME = False  # True after interruption; keep the same configuration
if RESUME and not LOCAL_RUN.exists():
    if not (DRIVE_RUN / "protocol.json").exists():
        raise FileNotFoundError("No backup protocol to restore")
    shutil.copytree(DRIVE_RUN, LOCAL_RUN)

In [ ]:
# Initial smoke configuration; this is not the expanded H1 experiment.
UPDATES = 2
TRIALS_PER_UPDATE = 4
STEPS = 4  # exploration budget K
ASSESSMENT_STEPS = 8  # assessment/control horizon H
HIDDEN_SIZE = 32
BATCH_SIZE = 4
SEEDS = [7]
CHECKPOINT_EVERY = 1
STOP_AFTER_UPDATE = None  # e.g. 1 to exercise restart; UPDATES remains the final target
cmd = [sys.executable, "-B", "-m", "mamrl.meta_exploration",
       "--catalog", "relational", "--opponent-suite", "mixed_learning",
       "--device", "cuda", "--batch-size", str(BATCH_SIZE),
       "--output-dir", str(LOCAL_RUN), "--backup-dir", str(DRIVE_RUN),
       "--updates", str(UPDATES), "--trials-per-update", str(TRIALS_PER_UPDATE),
       "--steps", str(STEPS), "--assessment-steps", str(ASSESSMENT_STEPS), "--hidden-size", str(HIDDEN_SIZE), "--reps", "1",
       "--checkpoint-every", str(CHECKPOINT_EVERY), "--seeds", *map(str, SEEDS)]
if RESUME:
    cmd.append("--resume")
if STOP_AFTER_UPDATE is not None:
    cmd.extend(["--stop-after-update", str(STOP_AFTER_UPDATE)])
subprocess.run(cmd, check=True)

In [ ]:
import json
from IPython.display import HTML, display
report = LOCAL_RUN / "report.html"
if report.exists():
    display(HTML(report.read_text()))
else:
    print("Paused/incomplete. Resume using the same settings.")
for path in sorted(LOCAL_RUN.glob("seed*/timing-*.json")):
    print(path.parent.name, json.loads(path.read_text()))
print("Persistent backup:", DRIVE_RUN)

Compare CPU/CUDA using separate run directories with identical settings, seeds and batch size.
Report actual gameplay steps/s, control costs and collection/optimization time. This implementation
batches neural inference/replay; environments and tabular learners still run on CPU. GPU speedup is
workload dependent. GPU and CPU need numerical-tolerance/repeated-run checks, not bitwise equality.
Do not run the current fixed 28-rule runner as if it were the future 45-run diversity study.